# 01 - Data profiling and duplicate analysis (Phases 1-2)
Read-only audit of the nine raw Olist files. Nothing is modified here. All logic lives in `src/profiling.py`; results are also saved to `reports/data_quality_report.xlsx` and `reports/duplicate_analysis.xlsx`.

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_columns', 40, 'display.max_colwidth', 90, 'display.max_rows', 80)
from src.profiling import run_profiling
res = run_profiling()

## Overview of the nine files

In [2]:
res['overview']

,dataset,file,rows,columns,missing_cells,missing_cells_pct,columns_with_missing,exact_duplicate_rows,primary_key,duplicate_primary_keys,memory_mb
0,orders,olist_orders_dataset.csv,99441,8,4908,0.617,3,0,order_id,0,55.6
1,order_items,olist_order_items_dataset.csv,112650,7,0,0.000,0,0,order_id + order_item_id,0,37.7
2,payments,olist_order_payments_dataset.csv,103886,5,0,0.000,0,0,order_id + payment_sequential,0,17.0
3,reviews,olist_order_reviews_dataset.csv,99224,7,145903,21.006,2,0,review_id + order_id,0,44.5
4,customers,olist_customers_dataset.csv,99441,5,0,0.000,0,0,customer_id,0,32.5
5,products,olist_products_dataset.csv,32951,9,2448,0.825,8,0,product_id,0,6.6
6,sellers,olist_sellers_dataset.csv,3095,4,0,0.000,0,0,seller_id,0,0.8
7,geolocation,olist_geolocation_dataset.csv,1000163,5,0,0.000,0,261831,(none),0,182.6
8,category_translation,product_category_name_translation.csv,71,2,0,0.000,0,0,product_category_name,0,0.0


## Column profile (columns with missing values, whitespace or parse failures)

In [3]:
res['column_profile'].query('missing > 0 or leading_trailing_whitespace > 0 or datetime_parse_failures > 0')

,dataset,column,raw_dtype,intended_type,non_null,missing,missing_pct,unique,unique_pct,is_constant,leading_trailing_whitespace,empty_strings,uppercase_chars,non_ascii_chars,datetime_parse_failures,example
4,orders,order_approved_at,string,datetime,99281,160,0.161,90733,91.243,False,0.0,0.0,0.0,0.0,0.0,2017-10-02 11:07:15
5,orders,order_delivered_carrier_date,string,datetime,97658,1783,1.793,81018,81.473,False,0.0,0.0,0.0,0.0,0.0,2017-10-04 19:55:00
6,orders,order_delivered_customer_date,string,datetime,96476,2965,2.982,95664,96.202,False,0.0,0.0,0.0,0.0,0.0,2017-10-10 21:25:13
23,reviews,review_comment_title,string,string,11568,87656,88.342,4527,4.562,False,1998.0,2.0,8053.0,2636.0,NaN,recomendo
24,reviews,review_comment_message,string,string,40977,58247,58.703,36159,36.442,False,9451.0,27.0,33968.0,22658.0,NaN,Recebi bem antes do prazo estipulado.
33,products,product_category_name,string,string,32341,610,1.851,73,0.222,False,0.0,0.0,0.0,0.0,NaN,perfumaria
34,products,product_name_lenght,float64,float64,32341,610,1.851,66,0.200,False,NaN,NaN,NaN,NaN,NaN,40.0
35,products,product_description_lenght,float64,float64,32341,610,1.851,2960,8.983,False,NaN,NaN,NaN,NaN,NaN,287.0
36,products,product_photos_qty,float64,float64,32341,610,1.851,19,0.058,False,NaN,NaN,NaN,NaN,NaN,1.0
37,products,product_weight_g,float64,float64,32949,2,0.006,2204,6.689,False,NaN,NaN,NaN,NaN,NaN,225.0


## Numerical statistics

In [4]:
res['numeric_stats']

,dataset,column,count,mean,median,std,min,p01,p25,p75,p99,max,skewness,zeros,negatives
0,order_items,order_item_id,112650,1.1978,1.0000,0.7051,1.0000,1.0000,1.0000,1.0000,4.0000,21.0000,7.5804,0,0
1,order_items,price,112650,120.6537,74.9900,183.6339,0.8500,9.9900,39.9000,134.9000,890.0000,6735.0000,7.9232,0,0
2,order_items,freight_value,112650,19.9903,16.2600,15.8064,0.0000,4.4198,13.0800,21.1500,84.5200,409.6800,5.6399,383,0
3,payments,payment_sequential,103886,1.0927,1.0000,0.7066,1.0000,1.0000,1.0000,1.0000,3.0000,29.0000,16.1801,0,0
4,payments,payment_installments,103886,2.8533,1.0000,2.6871,0.0000,1.0000,1.0000,4.0000,10.0000,24.0000,1.6551,2,0
5,payments,payment_value,103886,154.1004,100.0000,217.4941,0.0000,6.6900,56.7900,171.8375,1039.9165,13664.0800,9.2540,9,0
6,reviews,review_score,99224,4.0864,5.0000,1.3476,1.0000,1.0000,4.0000,5.0000,5.0000,5.0000,-1.3644,0,0
7,products,product_name_lenght,32341,48.4769,51.0000,10.2457,5.0000,20.0000,42.0000,57.0000,63.0000,76.0000,-0.9032,0,0
8,products,product_description_lenght,32341,771.4953,595.0000,635.1152,4.0000,84.0000,339.0000,972.0000,3286.2000,3992.0000,1.9621,0,0
9,products,product_photos_qty,32341,2.1890,1.0000,1.7368,1.0000,1.0000,1.0000,3.0000,8.0000,20.0000,2.1934,0,0


## Categorical distributions

In [5]:
d = res['categorical_distributions']
d[d.column.isin(['order_status', 'payment_type', 'review_score', 'customer_state'])]

,dataset,column,value,count,pct,note
0,orders,order_status,delivered,96478,97.02,
1,orders,order_status,shipped,1107,1.113,
2,orders,order_status,canceled,625,0.629,
3,orders,order_status,unavailable,609,0.612,
4,orders,order_status,invoiced,314,0.316,
5,orders,order_status,processing,301,0.303,
6,orders,order_status,created,5,0.005,
7,orders,order_status,approved,2,0.002,
54,payments,payment_type,credit_card,76795,73.922,
55,payments,payment_type,boleto,19784,19.044,


## Missing order timestamps follow the order status
Missingness is structural: an order that was never shipped has no carrier or delivery date.

In [6]:
res['orders_missing_by_status']

,order_status,orders,missing_order_approved_at,missing_order_delivered_carrier_date,missing_order_delivered_customer_date
0,delivered,96478,14,2,8
1,shipped,1107,0,0,1107
2,canceled,625,141,550,619
3,unavailable,609,0,609,609
4,invoiced,314,0,314,314
5,processing,301,0,301,301
6,created,5,5,5,5
7,approved,2,0,2,2


## Suspicious and inconsistent values

In [7]:
res['quality_issues'].query('affected_records > 0')

,dataset,column,issue,affected_records,severity,detail
1,orders,order_delivered_customer_date,status=delivered but delivery date missing,8,medium,contradiction between status and timestamp; cannot be imputed reliably
2,orders,order_delivered_carrier_date,status=delivered but carrier hand-over date missing,2,low,
3,orders,order_approved_at,status=delivered but approval timestamp missing,14,low,
4,orders,order_approved_at,approval missing (all statuses),160,info,canceled=141; delivered=14; created=5
5,orders,order_delivered_customer_date,non-delivered status but delivery date present,6,low,canceled=6
7,orders,order_delivered_carrier_date,carrier hand-over before approval,1359,low,plausible operationally (seller ships before payment clears) or timestamp lag
8,orders,order_delivered_carrier_date,carrier hand-over before purchase,166,medium,
9,orders,order_delivered_customer_date,delivered to customer before carrier hand-over,23,medium,
12,orders,order_purchase_timestamp,sparse edge months (<100 orders) at start/end of the period,25,info,range 2016-09-04 21:15:19 .. 2018-10-17 17:30:18; incomplete months distort trend/fore...
15,order_items,freight_value,freight = 0 (free shipping),383,info,legitimate


## Redundancy

In [8]:
res['redundancy']

,dataset,column,finding,recommendation
0,orders/customers,customer_id,customer_id is 1:1 with order_id (99441 ids for 99441 orders): it is an order-level su...,keep as join key; use customer_unique_id for customer-level analysis
1,customers/sellers,"*_city, *_state",city and state are derivable from the postal prefix via geolocation,"keep (cheap, avoids a join; geolocation does not cover every prefix)"
2,products,"product_name_lenght, product_description_lenght, product_photos_qty","listing metadata, always missing together with category (same 610 rows)",keep; low analytical value but harmless
3,category_translation,product_category_name_english,lookup table; fully redundant once merged into products,"merge English name into cleaned products, keep the lookup file"
4,geolocation,(rows),261831 exact duplicate rows; many points per postal prefix,deduplicate and aggregate to one row per prefix


## Phase 2 - duplicates
Primary and composite keys are unique everywhere. Repeating foreign keys are legitimate transactions (items, split payments, repeat customers) and are kept.

In [9]:
res['dup_summary']

,dataset,rows,exact_duplicate_rows,exact_duplicate_pct,primary_key,duplicate_primary_keys,proposed_action
0,orders,99441,0,0.000,order_id,0,no duplicates; nothing to remove
1,order_items,112650,0,0.000,order_id + order_item_id,0,no exact duplicates; repeated order_id / product rows are units purchased - keep all
2,payments,103886,0,0.000,order_id + payment_sequential,0,no exact duplicates; repeated order_id rows are split payments - keep all
3,reviews,99224,0,0.000,review_id + order_id,0,"no exact duplicates; keep all rows, de-duplicate to one review per order only in analy..."
4,customers,99441,0,0.000,customer_id,0,no duplicates; repeated customer_unique_id = repeat customers - keep all
5,products,32951,0,0.000,product_id,0,no duplicates; nothing to remove
6,sellers,3095,0,0.000,seller_id,0,no duplicates; nothing to remove
7,geolocation,1000163,261831,26.179,(none),0,"drop exact duplicate rows (carry no information), then aggregate to one row per postal..."
8,category_translation,71,0,0.000,product_category_name,0,no duplicates; nothing to remove


In [10]:
res['dup_key_checks']

,dataset,key,key_type,expectation,is_unique,distinct_keys,rows_sharing_a_key,surplus_rows,verdict
0,orders,order_id,primary key,must be unique,True,99441,0,0,OK
1,orders,customer_id,alternate key,1:1 with order_id in Olist,True,99441,0,0,OK
2,order_items,order_id + order_item_id,composite primary key,must be unique,True,112650,0,0,OK
3,order_items,order_id,foreign key,repeats legitimately (multi-item orders),False,98666,23787,13984,legitimate repetition - keep
4,order_items,order_id + product_id + seller_id,business key,repeats legitimately (quantity > 1),False,102425,17313,10225,legitimate repetition - keep
5,payments,order_id + payment_sequential,composite primary key,must be unique,True,103886,0,0,OK
6,payments,order_id,foreign key,repeats legitimately (split payments / vouchers),False,99440,7407,4446,legitimate repetition - keep
7,reviews,review_id + order_id,composite primary key,must be unique,True,99224,0,0,OK
8,reviews,review_id,candidate key,repeats: one review can cover several orders,False,98410,1603,814,legitimate repetition - keep
9,reviews,order_id,foreign key,repeats: an order can be reviewed more than once,False,98673,1098,551,legitimate repetition - keep


### Reviews: shared `review_id` and re-reviewed orders

In [11]:
res['dup_reviews_summary']

,metric,value
0,review rows,99224
1,distinct review_id,98410
2,distinct order_id,98673
3,review_ids attached to >1 order,789
4,...of which every row has identical score,789
5,...of which every row has identical message,789
6,orders with >1 review,547
7,...of which the reviews disagree on score,202
8,interpretation,content-identical review copied to each order of a multi-order purchase; and re-review...


### Geolocation duplicates

In [12]:
res['dup_geolocation_duplicates']

,metric,value
0,rows,1000163.0
1,exact duplicate rows,261831.0
2,rows after exact de-duplication,738332.0
3,distinct postal prefixes,19015.0
4,prefixes with a single row,1043.0
5,rows per prefix - p25,10.0
6,rows per prefix - median,29.0
7,rows per prefix - p75,66.5
8,rows per prefix - p95,180.0
9,rows per prefix - p99,353.0


### Formatting variants of categorical values

In [13]:
res['dup_formatting_variants']

,dataset,column,raw_distinct,normalised_distinct,redundant_variants,values_with_whitespace_padding,values_with_accents,values_with_uppercase
0,customers,customer_city,4119,4119,0,0,0,0
1,customers,customer_state,27,27,0,0,0,27
2,sellers,seller_city,611,606,5,0,2,0
3,sellers,seller_state,23,23,0,0,0,23
4,geolocation,geolocation_city,8011,5965,2046,1,2085,0
5,geolocation,geolocation_state,27,27,0,0,0,27
6,orders,order_status,8,8,0,0,0,0
7,payments,payment_type,5,5,0,0,0,0
8,products,product_category_name,73,73,0,0,0,0
9,category_translation,product_category_name_english,71,71,0,0,0,0
